# A Federated Learning Approach for Speech Anti-Spoofing and Deepfake Detection

---



This notebook walks through an end-to-end implementation of a privacy-preserving countermeasure against synthetic/forged speech. We’ll (1) prepare audio data and augmentations that mimic real telephony/media distortions, (2) extract robust self-supervised features with wav2vec 2.0, (3) classify with a lightweight Multi-Scale Dilation Attention Network (MDAN) backend, and (4) train the whole pipeline in a federated learning setup so data never leaves the client. We’ll then evaluate with EER/t-DCF across standard benchmarks.

---



Before getting into data preparation or building the model, I start by cloning Hemlata Tak’s **SSL_Anti-spoofing** repository. This repo has been foundational in the anti-spoofing community, and a large portion of the helper utilities, data-processing routines, RawBoost augmentation code, and even the training/evaluation patterns used in this notebook are adapted directly from her implementation. Pulling the repository here ensures I can rely on those well-tested components instead of re-implementing everything from scratch, and it keeps this notebook consistent with the methodology presented in her original work.


In [ ]:
import git
!git clone https://github.com/TakHemlata/SSL_Anti-spoofing

Cloning into 'SSL_Anti-spoofing'...
remote: Enumerating objects: 1579, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 1579 (delta 37), reused 29 (delta 29), pack-reused 1525 (from 1)
Receiving objects: 100% (1579/1579), 30.56 MiB | 20.62 MiB/s, done.
Resolving deltas: 100% (301/301), done.




---



After cloning the external repository, I move all of its contents into my working directory.  
This helps keep the project structure tidy and ensures that all modules, augmentation scripts, and utilities from the cloned repo are immediately importable without adjusting Python paths.  
The cell simply iterates through everything inside `SSL_Anti-spoofing` and relocates it into the main workspace.


In [ ]:
import shutil
import os

# The source directory
source_dir = '/kaggle/working/SSL_Anti-spoofing'
# Destination directory
destination_dir = "/kaggle/working/"

# Ensure destination directory exists
os.makedirs(destination_dir, exist_ok=True)

# Loop through all files and folders in the source directory
for item_name in os.listdir(source_dir):
    # Get full path of the source and destination
    source_item = os.path.join(source_dir, item_name)
    destination_item = os.path.join(destination_dir, item_name)

    # Move each item
    shutil.move(source_item, destination_item)

print(f"All files and folders moved from {source_dir} to {destination_dir}")

All files and folders moved from /kaggle/working/SSL_Anti-spoofing to /kaggle/working/




---



The next step cleans up a specific file inside the cloned Fairseq directory.  
Some versions of `indexed_dataset.py` bundled with external repos can cause import conflicts or outdated dependency issues—especially in notebook environments like Kaggle.  
To avoid those errors, I explicitly remove this file before proceeding.  
This ensures Fairseq loads correctly using the environment’s installed version rather than the repo’s bundled one.


In [ ]:
file_path = "/kaggle/working/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1/fairseq/data/indexed_dataset.py"

# Delete the file
try:
    os.remove(file_path)
    print(f"File {file_path} deleted successfully.")
except FileNotFoundError:
    print(f"The file {file_path} does not exist.")
except PermissionError:
    print("You do not have permission to delete this file.")

File /kaggle/working/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1/fairseq/data/indexed_dataset.py deleted successfully.




---



Since the previous step removed a problematic Fairseq file, here I replace it with a clean, compatible version that I’ve already prepared and stored in my Kaggle input directory.  
This ensures that Fairseq’s dataset handling behaves correctly during training—especially important because wav2vec 2.0 and other SSL models rely on Fairseq internals.  
The cell simply copies the known-good `indexed_dataset.py` into the Fairseq folder inside the working directory.


In [ ]:
source_path = "/kaggle/input/float32/indexed_dataset.py"
destination_path = "/kaggle/working/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1/fairseq/data/indexed_dataset.py"

# Copy the file
shutil.copy2(source_path, destination_path)

print(f"Copied {source_path} to {destination_path}")

Copied /kaggle/input/float32/indexed_dataset.py to /kaggle/working/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1/fairseq/data/indexed_dataset.py




---



Now that the Fairseq directory has the corrected file structure, I install it in **editable mode**.  
Using `pip install --editable` lets Python import Fairseq directly from the local folder, meaning any adjustments or patches inside this directory take effect immediately without needing to reinstall.  
This is important because wav2vec 2.0 (and other SSL models) depend on Fairseq internals, so having a clean, editable install avoids version conflicts and makes debugging easier.


In [ ]:
!cd fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1 && pip install --editable ./

Obtaining file:///kaggle/working/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 11.7 MB/s eta 0:00:00
  Building editable for fairseq (pyproject.toml) ... done
  Created wheel for fairseq: filename=fairseq-1.0.0a0+4acaa61-0.editable-cp310-cp310-linux_x86_64.whl size=9471 sha256=6fcc9943ae8c4733de5d13ce84fb27f04f6389686e7653c



---



With Fairseq set up, I now install all remaining dependencies listed in the project’s `requirements.txt`.  
This usually includes audio libraries, model toolkits, and utility packages needed for training, augmentation, and evaluation.  
Running this step ensures the environment matches the expected setup of the original implementation, preventing compatibility issues later in the notebook.


In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.1/213.1 kB 4.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.8 MB/s eta 0:00:00a 0:00:01
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
  Attempting uninstall: tensorboardX
    Found existing installation: tensorboardX 2.6.2.2
    Uninstalling tensorboardX-2.6.2.2:
      Successfully uninstalled tensorboardX-2.6.2.2
  Attempting uninstall: librosa
    Found existing installation: librosa 0.10.2.post1
    Uninstalling librosa-0.10.2.post1:
      Successfully uninstalled librosa-0.10.2.post1




---



Here I set up the Python environment for the rest of the notebook.  
First, I manually append the Fairseq directory to `sys.path` so Python can import its modules without issues. Then I load all the core libraries needed for training: PyTorch layers, datasets, utilities, argument parsing, YAML config handling, and NumPy.

I also import project-specific modules:  
- dataset builders for ASVspoof 2019 and 2021,  
- RawBoost feature processing,  
- the main `Model` class (wav2vec frontend + MDAN backend),  
- TensorBoard logging,  
- and a helper to set reproducible seeds.

This cell essentially prepares all the building blocks the training loop will rely on.


In [ ]:
import sys
sys.path.append('/kaggle/working/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1')
import argparse
import sys
import os
import numpy as np
import torch
from torch import nn
from torch import Tensor
from torch.utils.data import DataLoader
import yaml
from data_utils_SSL import genSpoof_list,Dataset_ASVspoof2019_train,Dataset_ASVspoof2021_eval, process_Rawboost_feature
from model import Model
from tensorboardX import SummaryWriter
from core_scripts.startup_config import set_random_seed



---



In this cell I put together the core training/evaluation utilities. I start by enabling **mixed-precision** with PyTorch AMP (using `autocast` and a `GradScaler`) to speed things up on CUDA while keeping numerical stability. Then I define:

- **`evaluate_accuracy`**: runs the model in `eval()` mode with gradients off, computes a **weighted cross-entropy** loss (class weights `[0.1, 0.9]`), and supports an optional `max_samples` cap so I can do quick sanity checks on a subset without burning time.
- **`train_epoch`**: switches to `train()` mode, streams batches with a `tqdm` progress bar, moves tensors to GPU with `non_blocking=True`, does the forward pass under `autocast('cuda')`, and performs the backward/update steps via the **GradScaler**. I accumulate a sample-weighted running loss and return the epoch average.

This gives me clean, reproducible building blocks for the federated rounds and the centralized baseline alike.


In [ ]:
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast # Updated import for GradScaler

# Initialize GradScaler for mixed precision on CUDA
scaler = GradScaler(device='cuda')

def evaluate_accuracy(dev_loader, model, device, max_samples=None):
    val_loss = 0.0
    num_total = 0
    model.eval()
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor([0.1, 0.9]).to(device))

    with torch.no_grad():  # Disable gradient calculation for evaluation
        for batch_x, batch_y in dev_loader:
            batch_size = batch_x.size(0)

            # Stop if max_samples limit is reached
            if max_samples and num_total + batch_size > max_samples:
                batch_size = int(max_samples - num_total)  # Convert to integer to meet slicing requirement
                batch_x = batch_x[:batch_size]
                batch_y = batch_y[:batch_size]

            num_total += batch_size

            batch_x = batch_x.to(device)
            batch_y = batch_y.view(-1).type(torch.int64).to(device)

            batch_out = model(batch_x)
            batch_loss = criterion(batch_out, batch_y)

            val_loss += (batch_loss.item() * batch_size)

            if max_samples and num_total >= max_samples:
                break  # Stop processing further batches once max_samples is reached

    val_loss /= num_total
    return val_loss

def train_epoch(train_loader, model, optimizer, device):
    running_loss = 0.0
    num_total = 0.0
    model.train()

    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor([0.1, 0.9]).to(device))

    # Use tqdm for tracking batch progress within each epoch
    batch_iterator = tqdm(train_loader, desc="Training", leave=True)
    for batch_x, batch_y in batch_iterator:
        batch_size = batch_x.size(0)
        num_total += batch_size

        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.view(-1).type(torch.int64).to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast('cuda'):  # Mixed precision for faster training
            batch_out = model(batch_x)
            batch_loss = criterion(batch_out, batch_y)

        # Backward pass with mixed precision
        scaler.scale(batch_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate loss
        running_loss += (batch_loss.item() * batch_size)

        # Update tqdm description with batch loss
        batch_iterator.set_description(f"Training - Batch Loss: {batch_loss.item():.4f}")

    running_loss /= num_total
    return running_loss



---



Here I centralize all the knobs for this run into a single config dictionary.  
It captures the usual training hyperparameters (batch size, epochs, `lr=1e-6`, weight decay), logging/eval flags, and the path to a checkpoint I want to load. I’m also fixing the random seed for reproducibility and specifying the **LA** track by default.

Below that are the **RawBoost** augmentation parameters, grouped by type:
- **LnL** (convolutive/channel) noise controls the number and shape of notch filters,
- **ISD** (impulsive) noise toggles short spikes,
- **SSI** (stationary) sets a colored-noise SNR range.

Finally, I wrap the dict with `SimpleNamespace` so I can access fields as `args.foo` instead of `args['foo']`, which keeps the rest of the code clean.


In [ ]:
# args dictionary with hardcoded values
args = {
    'batch_size': 16,
    'num_epochs': 20,
    'lr': 0.000001,
    'weight_decay': 0.0001,
    'loss': 'weighted_CCE',
    'seed': 1234,  # Random seed for reproducibility
    'model_path': '/kaggle/input/weights-f4/epoch_3.pth',  # Path to a model checkpoint
    'comment': None,  # Description for the saved model
    'track': 'LA',  # Dataset track; options are 'LA', 'PA', 'DF'
    'eval_output': 'eval_CM_scores_file_SSL_LA.txt',  # Path to save evaluation results
    'eval': True,  # Boolean for evaluation mode
    'is_eval': True,  # Boolean for evaluation dataset
    'eval_part': 0,  # Part of evaluation to process
    'cudnn_deterministic_toggle': False,  # Use deterministic CuDNN behavior
    'cudnn_benchmark_toggle': True,  # Use CuDNN benchmark for faster runtime

    # RawBoost data augmentation options
    'algo': 5,  # RawBoost algorithm selection

    # LnL_convolutive_noise parameters
    'nBands': 5,  # Number of notch filters
    'minF': 20,  # Minimum center frequency of notch filter
    'maxF': 8000,  # Maximum center frequency of notch filter
    'minBW': 100,  # Minimum bandwidth of filter
    'maxBW': 1000,  # Maximum bandwidth of filter
    'minCoeff': 10,  # Minimum filter coefficients
    'maxCoeff': 100,  # Maximum filter coefficients
    'minG': 0,  # Minimum gain factor of linear component
    'maxG': 0,  # Maximum gain factor of linear component
    'minBiasLinNonLin': 5,  # Minimum gain difference between linear/non-linear components
    'maxBiasLinNonLin': 20,  # Maximum gain difference between linear/non-linear components
    'N_f': 5,  # Order of non-linearity (1 means only linear)

    # ISD_additive_noise parameters
    'P': 10,  # Max number of uniformly distributed samples in [%]
    'g_sd': 2,  # Gain parameter for additive noise

    # SSI_additive_noise parameters
    'SNRmin': 10,  # Minimum SNR for colored noise
    'SNRmax': 40  # Maximum SNR for colored noise
}

from types import SimpleNamespace
args = SimpleNamespace(**args)



---



Before starting any training or evaluation, I create a dedicated `models` folder if it doesn't already exist—this is where checkpoints and logs will be saved later.  
Next, I call `set_random_seed(...)` to fix all relevant seeds (Python, NumPy, PyTorch, CUDA) so the results are reproducible across runs.

Finally, I read the selected dataset track from `args.track` and sanity-check that it’s one of the supported options: **LA**, **PA**, or **DF**. This prevents accidental typos from triggering confusing downstream errors.


In [ ]:
if not os.path.exists('models'):
    os.mkdir('models')
set_random_seed(args.seed, args)
track = args.track
assert track in ['LA', 'PA','DF'], 'Invalid track given'

cudnn_deterministic set to False
cudnn_benchmark set to True




---



This cell defines the **entire model stack** used in the federated learning system:  
a wav2vec 2.0–based frontend and the Multi-Scale Dilation Attention backend (a PyTorch implementation of the MDAN described in the paper).  
It’s essentially the heart of the model.


### **1) SSLModel — wav2vec 2.0 Frontend**
I start by loading the **XLSR-300M wav2vec 2.0** checkpoint via Fairseq.  
Only one model from the ensemble is used, and it’s immediately moved to the chosen device.  
The `extract_feat()` method cleanly handles input shapes and returns feature embeddings of shape:




These embeddings serve as the high-level speech representation.


### **2) PSFAN_Backend — Multi-Scale Dilation Attention Network**
This is the **backend classifier**, matching the architecture from the paper:

✔ Four Conv1D blocks with **dilations = 1, 2, 3, 4**  
✔ Each block uses:
- a dilated convolution  
- a LeakyReLU  
- a 1×1 → 3×1 → 1×1 refinement stack  
- a gated attention map via **sigmoid**  
- a residual-style fusion (features + attention-weighted features)  
- **Global Average Pooling (GAP)** per block  

✔ Output sizes per block:  
- Block 1 → 64  
- Block 2 → 64  
- Block 3 → 128  
- Block 4 → 128  

✔ Final vector:  



This is passed to a small fully connected layer (384 → 16) and then to a final classifier (16 → 2 classes).

This backend is lightweight but captures **multi-scale temporal structure**, matching the MDAN design in the paper.


### **3) Model — Combining Frontend + Backend**
The full model:

- Extracts wav2vec features  
- Reduces them from **1024 → 128** channels via a linear layer  
- Transposes the tensor for Conv1D backend  
- Passes through PSFAN/MDAN backend  
- Outputs class logits (bonafide vs spoof)

All components are moved to the correct device, ensuring consistent GPU execution.

In short, this cell builds the complete architecture used for federated training:  
**wav2vec 2.0 → linear projection → dilated-attention backend → softmax logits**.


In [ ]:
import random
from typing import Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import fairseq

class SSLModel(nn.Module):
    def __init__(self, device):
        super(SSLModel, self).__init__()

        cp_path = '/kaggle/input/xlsr2-300m/xlsr2_300m.pt'   # Path to pre-trained model
        model, cfg, task = fairseq.checkpoint_utils.load_model_ensemble_and_task([cp_path])
        self.model = model[0].to(device)  # Move the model to the specified device only once
        self.device = device
        self.out_dim = 1024

    def extract_feat(self, input_data):
        # Ensure input is on the correct device
        input_data = input_data.to(self.device)

        # Adjust input shape to (batch, length) if necessary
        input_tmp = input_data[:, :, 0] if input_data.ndim == 3 else input_data

        # Extract features [batch, length, dim]
        emb = self.model(input_tmp, mask=False, features_only=True)['x']
        return emb


import torch
import torch.nn as nn

class PSFAN_Backend(nn.Module):
    def __init__(self, input_channels=128, num_classes=2):
        super(PSFAN_Backend, self).__init__()

        self.leakyrelu = nn.LeakyReLU(0.02)

        # Block 1 (dilation=1), out_channels=64
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=3, dilation=1, padding=1)
        self.conv1x1_1 = nn.Conv1d(64, 64, kernel_size=1)
        self.conv3x3_1 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv1x1_2 = nn.Conv1d(64, 64, kernel_size=1)
        self.attention1 = nn.Sigmoid()
        self.gap1 = nn.AdaptiveAvgPool1d(1)

        # Block 2 (dilation=2), out_channels=64
        self.conv2 = nn.Conv1d(64, 64, kernel_size=3, dilation=2, padding=2)
        self.conv1x1_3 = nn.Conv1d(64, 64, kernel_size=1)
        self.conv3x3_2 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv1x1_4 = nn.Conv1d(64, 64, kernel_size=1)
        self.attention2 = nn.Sigmoid()
        self.gap2 = nn.AdaptiveAvgPool1d(1)

        # Block 3 (dilation=3), out_channels=128
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, dilation=3, padding=3)
        self.conv1x1_5 = nn.Conv1d(128, 128, kernel_size=1)
        self.conv3x3_3 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv1x1_6 = nn.Conv1d(128, 128, kernel_size=1)
        self.attention3 = nn.Sigmoid()
        self.gap3 = nn.AdaptiveAvgPool1d(1)

        # Block 4 (dilation=4), out_channels=128
        self.conv4 = nn.Conv1d(128, 128, kernel_size=3, dilation=4, padding=4)
        self.conv1x1_7 = nn.Conv1d(128, 128, kernel_size=1)
        self.conv3x3_4 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv1x1_8 = nn.Conv1d(128, 128, kernel_size=1)
        self.attention4 = nn.Sigmoid()
        self.gap4 = nn.AdaptiveAvgPool1d(1)

        # Fully connected layers:
        # After concatenation: 64 + 64 + 128 + 128 = 384
        # Single Dense layer with 16 units, followed by output layer
        self.fc_concat = nn.Linear(384, 16)
        self.fc_out = nn.Linear(16, num_classes)

    def forward(self, x):
        # Block 1
        x1_conv = self.conv1(x)
        x1_conv = self.leakyrelu(x1_conv)  # LeakyReLU after first conv
        x1_feat = self.conv1x1_2(self.conv3x3_1(self.conv1x1_1(x1_conv)))
        x1_attention_map = self.attention1(x1_feat)
        x1_attention = x1_attention_map * x1_conv
        x1_out = x1_feat + x1_attention
        x1_gap = self.gap1(x1_out).squeeze(-1)  # (B,64)

        # Block 2 (takes x1_conv as input)
        x2_conv = self.conv2(x1_conv)
        x2_conv = self.leakyrelu(x2_conv)
        x2_feat = self.conv1x1_4(self.conv3x3_2(self.conv1x1_3(x2_conv)))
        x2_attention_map = self.attention2(x2_feat)
        x2_attention = x2_attention_map * x2_conv
        x2_out = x2_feat + x2_attention
        x2_gap = self.gap2(x2_out).squeeze(-1)  # (B,64)

        # Block 3 (takes x2_conv as input)
        x3_conv = self.conv3(x2_conv)
        x3_conv = self.leakyrelu(x3_conv)
        x3_feat = self.conv1x1_6(self.conv3x3_3(self.conv1x1_5(x3_conv)))
        x3_attention_map = self.attention3(x3_feat)
        x3_attention = x3_attention_map * x3_conv
        x3_out = x3_feat + x3_attention
        x3_gap = self.gap3(x3_out).squeeze(-1)  # (B,128)

        # Block 4 (takes x3_conv as input)
        x4_conv = self.conv4(x3_conv)
        x4_conv = self.leakyrelu(x4_conv)
        x4_feat = self.conv1x1_8(self.conv3x3_4(self.conv1x1_7(x4_conv)))
        x4_attention_map = self.attention4(x4_feat)
        x4_attention = x4_attention_map * x4_conv
        x4_out = x4_feat + x4_attention
        x4_gap = self.gap4(x4_out).squeeze(-1)  # (B,128)

        # Concatenate all GAP outputs
        x_concat = torch.cat([x1_gap, x2_gap, x3_gap, x4_gap], dim=1)  # (B,384)

        # Single Dense layer with optional LeakyReLU
        x = self.fc_concat(x_concat)
        x = self.leakyrelu(x)

        # Final output layer
        output = self.fc_out(x)  # (B, num_classes)

        return output



class Model(nn.Module):
    def __init__(self, args, device):
        super(Model, self).__init__()
        self.device = device

        # wav2vec 2.0 front-end remains unchanged
        self.ssl_model = SSLModel(self.device)
        self.LL = nn.Linear(self.ssl_model.out_dim, 128).to(device)  # Reduces dimensionality to 128 for compatibility

        # PSFAN backend with Conv1D for multi-scale feature extraction
        self.backend = PSFAN_Backend(input_channels=128, num_classes=2).to(device)

    def forward(self, x):
        # Move input to the same device as model
        x = x.to(self.device)

        # wav2vec 2.0 feature extraction
        x_ssl_feat = self.ssl_model.extract_feat(x)
        x = self.LL(x_ssl_feat)  # Dimensionality reduction to 128 channels
        x = x.transpose(1, 2)  # Reshape to (batch, features, timesteps) for Conv1D format

        # Backend processing for classification
        output = self.backend(x)
        return output



---



Here I assemble the **global model** from three client checkpoints—this is the FedAvg-style aggregation used in the paper. I first build a clean model instance, set the device, and print the parameter count so I know what I’m working with. Then I load each client’s `state_dict` from disk and **average the weights key-by-key** to produce a single aggregated checkpoint. If any file is missing, I print a warning so it’s obvious what went wrong. Finally, I load the averaged weights into the model and initialize an Adam optimizer with the configured learning rate and weight decay. This gives me a ready-to-train (or evaluate) global model that reflects contributions from all three clients.


In [ ]:
model_path1 = '/kaggle/input/weights-f2-round/f2_t_epoch_1.pth' # Path to model weights from client 1
model_path2 = '/kaggle/input/weights-f2-round/f2_d_epoch_3.pth'  # Path to model weights from client 2
model_path3 = '/kaggle/input/weights-f2-round/f2_fr_epoch_2.pth' # Path to model weights from client 3

# Model tag and save path setup
model_tag = 'model_{}_{}_{}_{}_{}'.format(
    track, args.loss, args.num_epochs, args.batch_size, args.lr)
if args.comment:
    model_tag = model_tag + '_{}'.format(args.comment)
model_save_path = os.path.join('models', model_tag)

# Ensure model save directory exists
if not os.path.exists(model_save_path):
    os.mkdir(model_save_path)

# Set device to GPU if available, else CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device: {}'.format(device))

# Initialize the model architecture (without loading weights)
model = Model(args, device).to(device)
nb_params = sum([param.view(-1).size()[0] for param in model.parameters()])
print('nb_params:', nb_params)

# Function to load model weights from a given path
def load_model_weights(path, device):
    if os.path.exists(path):
        return torch.load(path, map_location=device)
    else:
        print(f"Warning: Model path {path} does not exist.")
        return None

# Load weights from each client model
state_dict1 = load_model_weights(model_path1, device)
state_dict2 = load_model_weights(model_path2, device)
state_dict3 = load_model_weights(model_path3, device)

# Ensure all state_dicts are loaded successfully
if state_dict1 and state_dict2 and state_dict3:
    # Aggregate weights by averaging
    aggregated_state_dict = {}
    for key in state_dict1.keys():
        # Average weights from the three clients
        aggregated_state_dict[key] = (state_dict1[key] + state_dict2[key] + state_dict3[key]) / 3.0

    # Load the aggregated weights into the model
    del state_dict1, state_dict2, state_dict3
    model.load_state_dict(aggregated_state_dict)
    del aggregated_state_dict
    print("Aggregated model weights loaded successfully.")
else:
    print("Error: Failed to load weights from all clients. Check file paths.")

# Set up the optimizer with the aggregated model
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

Device: cuda


/kaggle/working/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1/fairseq/checkpoint_utils.py:313: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(f, map_loc

nb_params: 317844914


/tmp/ipykernel_30/851587500.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(path, map_location=device)


Aggregated model weights loaded successfully.




---



Here I prepare the **training and validation dataloaders** for the ASVspoof2019 LA track.

### **Training Loader**
1. I start by calling `genSpoof_list()` with the LA training protocol file.  
   This returns:
   - `d_label_trn`: a dictionary mapping each utterance ID to its label (0 = bonafide, 1 = spoof).  
   - `file_train`: a list of utterance IDs to load.

2. I print how many training trials there are so I know the split looks correct.

3. I then create a `Dataset_ASVspoof2019_train` instance with:
   - the protocol list  
   - labels  
   - dataset base path  
   - and the RawBoost augmentation algorithm selected in `args.algo`.

4. I wrap this dataset in a `DataLoader`, enabling:
   - `shuffle=True` for random sampling during training  
   - 4 workers for faster I/O  
   - `pin_memory=True` to speed up transfers to CUDA  
   - `drop_last=True` to keep batch sizes consistent

After creating `train_loader`, I free the raw dataset objects to save memory.



### **Validation Loader**
I repeat a similar process for the validation split:
- Call `genSpoof_list()` again (with `is_train=False`).  
- Build a validation dataset with **no shuffling** so results are consistent.  
- Wrap it in a `DataLoader`, keeping `pin_memory=True` but leaving `shuffle=False`.

Finally, I clear unused variables again to save memory.

This gives me the two core dataloaders needed for training epochs and evaluation loops.


In [ ]:
# Define train dataloader
d_label_trn, file_train = genSpoof_list(
    dir_meta="/kaggle/input/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt",
    is_train=True, is_eval=False)
print('no. of training trials', len(file_train))

train_set = Dataset_ASVspoof2019_train(
    args, list_IDs=file_train, labels=d_label_trn,
    base_dir="/kaggle/input/asvpoof-2019-dataset/LA/LA/ASVspoof2019_LA_train/",
    algo=args.algo)

train_loader = DataLoader(
    train_set, batch_size=args.batch_size, num_workers=4, shuffle=True,
    drop_last=True, pin_memory=True)  # Pin memory for faster data transfer to GPU

del train_set, d_label_trn

# Define validation dataloader
d_label_dev, file_dev = genSpoof_list(
    dir_meta="",
    is_train=False, is_eval=False)
print('no. of validation trials', len(file_dev))

dev_set = Dataset_ASVspoof2019_train(
    args, list_IDs=file_dev, labels=d_label_dev,
    base_dir="",
    algo=args.algo)

dev_loader = DataLoader(
    dev_set, batch_size=args.batch_size, num_workers=4, shuffle=False,
    pin_memory=True)  # Pin memory for faster data transfer to GPU

del dev_set, d_label_dev

no. of training trials 25380
no. of validation trials 71237




---



I’m adding a compact dataset wrapper for the **Fake-or-Real (FoR)** split so I can quickly sanity-check the pipeline outside the ASVspoof protocol files. A tiny helper `pad_repeat` enforces the paper’s fixed ~4 s length (64,600 samples): long clips are cropped, short clips are loop-padded then trimmed. The `FakeRealDataset` uses `librosa` to resample to **16 kHz**, applies our **RawBoost** augmentation (`process_Rawboost_feature`), converts to tensors, and returns `(waveform, label)` pairs where **1 = bonafide** and **0 = spoof**. Finally, I wrap it with a DataLoader that mirrors the earlier hyperparameters (batch size, workers, pin memory) so it drops into the same training loop without surprises.


In [ ]:
# ----------------------------------------------
# 1. Imports
# ----------------------------------------------
from pathlib import Path
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset, DataLoader

# ----------------------------------------------
# 2. Helper: fixed‑length crop / repeat
# ----------------------------------------------
def pad_repeat(x: np.ndarray, max_len: int = 64_600) -> np.ndarray:
    """Crop if longer than max_len, otherwise repeat until max_len."""
    if x.shape[0] >= max_len:                 # crop
        return x[:max_len]
    # repeat
    n_repeats = max_len // x.shape[0] + 1
    return np.tile(x, n_repeats)[:max_len]


# ----------------------------------------------
# 3. Dataset
# ----------------------------------------------
class FakeRealDataset(Dataset):
    """
    • real_dir : folder containing 'bonafide' wav files
    • fake_dir : folder containing 'spoof'    wav files
    • args     : same Namespace you already pass around
    • algo     : which RawBoost algorithm to apply
    """
    def __init__(self,
                 real_dir: str,
                 fake_dir: str,
                 args,
                 algo: str,
                 max_len: int = 64_600):

        self.args     = args
        self.algo     = algo
        self.max_len  = max_len

        # Collect <Path, label> pairs  (1 = real, 0 = fake)
        self.samples = []

        for p in sorted(Path(real_dir).glob("*.wav")):
            self.samples.append((p, 1))
        for p in sorted(Path(fake_dir).glob("*.wav")):
            self.samples.append((p, 0))

    # mandatory Dataset API ------------------------------------------
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        wav_path, label = self.samples[idx]

        # 1) load & resample to 16 kHz
        x, fs = librosa.load(wav_path, sr=16_000)

        # 2) RawBoost augmentation  (user‑supplied function)
        x = process_Rawboost_feature(x, fs, self.args, self.algo)

        # 3) force fixed length
        x = pad_repeat(x, self.max_len)

        # 4) to torch tensor
        x = torch.tensor(x, dtype=torch.float32)

        return x, label

# ----------------------------------------------
# 4. Instantiate
# ----------------------------------------------
REAL_DIR  = "/kaggle/input/the-fake-or-real-dataset/for-norm/for-norm/validation/real"
FAKE_DIR  = "/kaggle/input/the-fake-or-real-dataset/for-norm/for-norm/validation/fake"

train_set = FakeRealDataset(
    real_dir=REAL_DIR,
    fake_dir=FAKE_DIR,
    args=args,
    algo=args.algo)

train_loader = DataLoader(
    train_set,
    batch_size=args.batch_size,   # keep same hyper‑param interface
    shuffle=True,
    num_workers=4,
    drop_last=True,
    pin_memory=True)              # fast CPU→GPU transfer


In [ ]:
# Grab one batch and inspect its shape
for data, target in train_loader:
    print(f"data shape  : {data.shape}")    # e.g. torch.Size([64, 3, 32, 32])
    print(f"target shape: {target.shape}")  # e.g. torch.Size([64])
    print(target)
    break

/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


data shape  : torch.Size([16, 64600])
target shape: torch.Size([16])
tensor([0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1])




---



To wrap up the evaluation workflow, I add a clean utility for computing **Equal Error Rate (EER)** — the primary metric used in ASVspoof and deepfake-detection research. The function loops over a dataloader in `eval()` mode, collects the model’s **bonafide logits** (output[:,1]) and ground-truth labels, and then uses `sklearn`’s `roc_curve` to generate the full detection curve. The EER is the point where **FPR = FNR**, so I simply find the threshold where their difference is minimized. Finally, I print and return the EER in percent. At the end, I move the model to the right device and call the function on the Fake-or-Real loader to get a quick sanity-check score.


In [ ]:
import numpy as np
from sklearn.metrics import roc_curve
import torch

def evaluate_eer(model: torch.nn.Module,
                 data_loader: torch.utils.data.DataLoader,
                 device: torch.device | str = "cpu") -> float:
    """
    Evaluate Equal Error Rate (EER) for a binary spoof-detection model.

    Notes
    -----
    * `label = 1`  → bonafide / real
    * `label = 0`  → spoof  / fake
    * Model output is `(B,2)` logits in the current architecture.
      We use `logit_bonafide = output[:,1]` as the decision score.
    """

    model.eval()
    scores, labels = [], []

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            out = model(x)                 # (B,2) logits
            if out.dim() == 2 and out.size(1) == 2:
                s = out[:, 1]                              # bonafide score
            else:                                           # fallback: (B,) or (B,1)
                s = out.squeeze()

            scores.append(s.cpu())
            labels.append(y.cpu())

    # --- stack & to numpy ---
    scores = torch.cat(scores).numpy()
    labels = torch.cat(labels).numpy()

    # --- ROC & EER ---
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1.0 - tpr

    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = 0.5 * (fpr[idx] + fnr[idx]) * 100.0

    print(f"EER ≈ {eer:.2f}%  @ threshold = {thresholds[idx]:.4f}")
    return eer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

eer_val = evaluate_eer(model, train_loader, device)

EER ≈ 1.47%  @ threshold = -5.9311




---



This is the main training loop. I open a TensorBoard writer, then iterate for `num_epochs`. Each epoch I:

1) **Train** on the current loader and capture the average loss.  
2) **Validate** on the dev set (capped at 10k samples to keep it snappy), wrapped in a `try/except` so a flaky batch doesn’t kill the run.  
3) **Log** both training and (if available) validation loss to TensorBoard under this run’s `model_tag`.  
4) **Checkpoint** the model weights every epoch to `models/<model_tag>/epoch_*.pth` so I can resume or compare later.

This keeps the loop robust, observable, and reproducible without adding too much boilerplate.


In [ ]:
num_epochs = args.num_epochs
writer = SummaryWriter('logs/{}'.format(model_tag))

for epoch in range(num_epochs):
    print('Starting epoch {}'.format(epoch))

    # Training step
    running_loss = train_epoch(train_loader, model, optimizer, device)

    # Validation step with error handling
    try:
        val_loss = evaluate_accuracy(dev_loader, model, device, max_samples=10000)
        writer.add_scalar('val_loss', val_loss, epoch)
        print('\nEpoch {} - Training Loss: {:.4f} - Validation Loss: {:.4f}'.format(epoch, running_loss, val_loss))
    except Exception as e:
        print(f"Warning: An error occurred during validation at epoch {epoch}: {e}")
        val_loss = None

    # Log training loss regardless of validation success
    writer.add_scalar('loss', running_loss, epoch)

    # Save the model state
    torch.save(model.state_dict(), os.path.join(model_save_path, f'epoch_{epoch}.pth'))

Starting epoch 0


Training:   0%|          | 0/1586 [00:00<?, ?it/s]/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Training - Batch Loss: 0.0001: 100%|██████████| 1586/1586 [26:41<00:00,  1.00s/it]/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Training - Batch Loss: 0.0001: 100%|██████████| 1586/1586 [26:41<00:00,  1.01s/it]



Epoch 0 - Training Loss: 0.0040 - Validation Loss: 0.0468
Starting epoch 1


Training - Batch Loss: 0.0002: 100%|██████████| 1586/1586 [26:25<00:00,  1.00it/s]



Epoch 1 - Training Loss: 0.0023 - Validation Loss: 0.0309
Starting epoch 2


...

Training - Batch Loss: 0.0000: 100%|██████████| 1586/1586 [25:57<00:00,  1.02it/s]



Epoch 5 - Training Loss: 0.0008 - Validation Loss: 0.0151
Starting epoch 6


Training - Batch Loss: 0.0000: 100%|██████████| 1586/1586 [26:01<00:00,  1.02it/s]



Epoch 6 - Training Loss: 0.0015 - Validation Loss: 0.0250
Starting epoch 7


Training - Batch Loss: 0.0003: 100%|██████████| 1586/1586 [26:05<00:00,  1.01it/s]




---



To generate submission-style scores for the **ASVspoof2021 LA eval** split, I define a small utility that streams the evaluation dataset without gradients and writes one line per utterance: `<utt_id> <score>`. The score I export is the model’s **Class-1 (bonafide)** logit for each file, which matches the convention used by many ASVspoof baselines. I keep the loader deterministic (`shuffle=False`) and use a moderately large batch size for faster throughput. I also open the output file once in `a+` mode to avoid repeated I/O overhead and print a simple batch counter so I can see progress in the notebook logs.

Below, I parse the official **eval trial list**, construct the 2021-LA evaluation dataset, and call the writer to produce `eval_CM_scores_file_SSL_LA.txt`. This creates a clean text file you can feed into downstream EER/t-DCF scripts provided by the challenge toolkit.


In [ ]:
from torch.utils.data import DataLoader

def produce_evaluation_file(dataset, model, device, save_path):
    data_loader = DataLoader(dataset, batch_size=64, shuffle=False, drop_last=False, num_workers = 4)
    model.eval()

    batch_count = 0  # Counter for the number of batches processed

    with torch.no_grad():  # Disable gradient calculations for faster inference
        with open(save_path, 'a+') as fh:  # Open the file once before the loop
            for batch_x, utt_id in data_loader:
                # Move input data to device
                batch_x = batch_x.to(device)

                # Get model predictions
                batch_out = model(batch_x)

                # Extract scores for Class 1 and move to CPU
                batch_score = batch_out[:, 1].data.cpu().numpy().ravel()

                # Write batch results to file
                for f, cm in zip(utt_id, batch_score):
                    fh.write(f"{f} {cm}\n")

                # Increment batch counter and print status
                batch_count += 1
                print(f"Processed batch {batch_count}")

    print(f"Scores saved to {save_path}")
if args.eval:
    file_eval = genSpoof_list(
        dir_meta="/kaggle/input/avsspoof-2021/ASVspoof2021_LA_eval/ASVspoof2021_LA_eval/ASVspoof2021.LA.cm.eval.trl.txt",
        is_train=False, is_eval=True)
    print('no. of eval trials', len(file_eval))
    eval_set = Dataset_ASVspoof2021_eval(
        list_IDs=file_eval,
        base_dir="/kaggle/input/avsspoof-2021/ASVspoof2021_LA_eval/ASVspoof2021_LA_eval/")
    produce_evaluation_file(eval_set, model, device, args.eval_output)



---



Finally, I compute the official **ASVspoof 2021 LA** metrics from the scores file we just produced. This script loads the organizers’ **ASV metadata and ASV scores**, derives the ASV operating point (EER/threshold), and then fuses that with our **CM (countermeasure) scores** to calculate the **minimum t-DCF** alongside CM **EER**. I also include a safety check that negates scores to catch the classic “label flipped” mistake—if negating improves t-DCF, it warns that the positive/negative class was likely swapped during training. The script validates input sizes/paths, res


In [ ]:
import sys, os.path
import numpy as np
import pandas
import eval_metric_LA as em
from glob import glob


submit_file = '/kaggle/working/eval_CM_scores_file_SSL_LA.txt'
truth_dir = '/kaggle/input/avsspoof-2021/LA-keys-full/keys/LA'
phase = 'eval'

asv_key_file = os.path.join(truth_dir, 'ASV/trial_metadata.txt')
asv_scr_file = os.path.join(truth_dir, 'ASV/ASVTorch_Kaldi/score.txt')
cm_key_file = os.path.join(truth_dir, 'CM/trial_metadata.txt')


Pspoof = 0.05
cost_model = {
    'Pspoof': Pspoof,  # Prior probability of a spoofing attack
    'Ptar': (1 - Pspoof) * 0.99,  # Prior probability of target speaker
    'Pnon': (1 - Pspoof) * 0.01,  # Prior probability of nontarget speaker
    'Cmiss': 1,  # Cost of tandem system falsely rejecting target speaker
    'Cfa': 10,  # Cost of tandem system falsely accepting nontarget speaker
    'Cfa_spoof': 10,  # Cost of tandem system falsely accepting spoof
}


def load_asv_metrics():
    # Load organizers' ASV scores
    asv_key_data = pandas.read_csv(asv_key_file, sep=' ', header=None)
    asv_scr_data = pandas.read_csv(asv_scr_file, sep=' ', header=None)[asv_key_data[7] == phase]
    idx_tar = asv_key_data[asv_key_data[7] == phase][5] == 'target'
    idx_non = asv_key_data[asv_key_data[7] == phase][5] == 'nontarget'
    idx_spoof = asv_key_data[asv_key_data[7] == phase][5] == 'spoof'

    # Extract target, nontarget, and spoof scores from the ASV scores
    tar_asv = asv_scr_data[2][idx_tar]
    non_asv = asv_scr_data[2][idx_non]
    spoof_asv = asv_scr_data[2][idx_spoof]
    eer_asv, asv_threshold = em.compute_eer(tar_asv, non_asv)
    [Pfa_asv, Pmiss_asv, Pmiss_spoof_asv, Pfa_spoof_asv] = em.obtain_asv_error_rates(tar_asv, non_asv, spoof_asv, asv_threshold)

    return Pfa_asv, Pmiss_asv, Pmiss_spoof_asv, Pfa_spoof_asv


def performance(cm_scores, Pfa_asv, Pmiss_asv, Pfa_spoof_asv, invert=False):
    bona_cm = cm_scores[cm_scores[5]=='bonafide']['1_x'].values
    spoof_cm = cm_scores[cm_scores[5]=='spoof']['1_x'].values

    if invert==False:
        eer_cm = em.compute_eer(bona_cm, spoof_cm)[0]
    else:
        eer_cm = em.compute_eer(-bona_cm, -spoof_cm)[0]

    if invert==False:
        tDCF_curve, _ = em.compute_tDCF(bona_cm, spoof_cm, Pfa_asv, Pmiss_asv, Pfa_spoof_asv, cost_model, False)
    else:
        tDCF_curve, _ = em.compute_tDCF(-bona_cm, -spoof_cm, Pfa_asv, Pmiss_asv, Pfa_spoof_asv, cost_model, False)

    min_tDCF_index = np.argmin(tDCF_curve)
    min_tDCF = tDCF_curve[min_tDCF_index]

    return min_tDCF, eer_cm


def eval_to_score_file(score_file, cm_key_file):
    Pfa_asv, Pmiss_asv, Pmiss_spoof_asv, Pfa_spoof_asv = load_asv_metrics()
    cm_data = pandas.read_csv(cm_key_file, sep=' ', header=None)
    submission_scores = pandas.read_csv(score_file, sep=' ', header=None, skipinitialspace=True)

    if len(submission_scores) != len(cm_data):
        print('CHECK: submission has %d of %d expected trials.' % (len(submission_scores), len(cm_data)))
        exit(1)

    # check here for progress vs eval set
    cm_scores = submission_scores.merge(cm_data[cm_data[7] == phase], left_on=0, right_on=1, how='inner')
    min_tDCF, eer_cm = performance(cm_scores, Pfa_asv, Pmiss_asv, Pfa_spoof_asv)

    out_data = "min_tDCF: %.4f\n" % min_tDCF
    out_data += "eer: %.2f\n" % (100*eer_cm)
    print(out_data, end="")

    # just in case that the submitted file reverses the sign of positive and negative scores
    min_tDCF2, eer_cm2 = performance(cm_scores, Pfa_asv, Pmiss_asv, Pfa_spoof_asv, invert=True)

    if min_tDCF2 < min_tDCF:
        print(
            'CHECK: we negated your scores and achieved a lower min t-DCF. Before: %.3f - Negated: %.3f - your class labels are swapped during training... this will result in poor challenge ranking' % (
            min_tDCF, min_tDCF2))

    if min_tDCF == min_tDCF2:
        print(
            'WARNING: your classifier might not work correctly, we checked if negating your scores gives different min t-DCF - it does not. Are all values the same?')

    return min_tDCF


if __name__ == "__main__":

    if not os.path.isfile(submit_file):
        print("%s doesn't exist" % (submit_file))
        exit(1)

    if not os.path.isdir(truth_dir):
        print("%s doesn't exist" % (truth_dir))
        exit(1)

    if phase != 'progress' and phase != 'eval' and phase != 'hidden_track':
        print("phase must be either progress, eval, or hidden_track")
        exit(1)

    _ = eval_to_score_file(submit_file, cm_key_file)

min_tDCF: 0.2672
eer: 2.92




---



I reuse the same score-dumping helper, but this time point it at the **ASVspoof2021 DF (DeepFake)** evaluation split. The function streams the eval set without gradients, grabs the **bonafide logit** (column 1) for each utterance, and appends `<utt_id> <score>` lines to the output file. I keep the loader deterministic and print a simple “Processed batch *k*” tracker so I can gauge progress as it runs.


In [ ]:
from torch.utils.data import DataLoader

def produce_evaluation_file(dataset, model, device, save_path):
    data_loader = DataLoader(dataset, batch_size=64, shuffle=False, drop_last=False, num_workers = 4)
    model.eval()

    batch_count = 0  # Counter for the number of batches processed

    with torch.no_grad():  # Disable gradient calculations for faster inference
        with open(save_path, 'a+') as fh:  # Open the file once before the loop
            for batch_x, utt_id in data_loader:
                # Move input data to device
                batch_x = batch_x.to(device)

                # Get model predictions
                batch_out = model(batch_x)

                # Extract scores for Class 1 and move to CPU
                batch_score = batch_out[:, 1].data.cpu().numpy().ravel()

                # Write batch results to file
                for f, cm in zip(utt_id, batch_score):
                    fh.write(f"{f} {cm}\n")

                # Increment batch counter and print status
                batch_count += 1
                print(f"Processed batch {batch_count}")

    print(f"Scores saved to {save_path}")
if args.eval:
    file_eval = genSpoof_list(
        dir_meta="/kaggle/input/avsspoof-2021/ASVspoof2021_DF_eval_part00/ASVspoof2021_DF_eval/ASVspoof2021.DF.cm.eval.trl.txt",
        is_train=False, is_eval=True)
    print('no. of eval trials', len(file_eval))
    eval_set = Dataset_ASVspoof2021_eval(
        list_IDs=file_eval,
        base_dir="/kaggle/input/avsspoof-2021/ASVspoof2021_DF_eval_part00/ASVspoof2021_DF_eval/")
    produce_evaluation_file(eval_set, model, device, args.eval_output)



---



Now I switch over to the **ASVspoof2021 DF (DeepFake)** official evaluation. This helper mirrors the LA evaluator but is simpler: it only computes **EER** for the countermeasure (no ASV fusion or t-DCF here). I load the DF ground-truth metadata, merge it with our submission scores by utterance ID, warn if we’re missing any trials, split scores into bonafide vs. spoof, and then call the standard `compute_eer`. The script also sanity-checks file/dir existence and validates the chosen `phase` before running.


In [ ]:
import sys, os.path
import numpy as np
import pandas
import eval_metrics_DF as em

submit_file = '/kaggle/working/eval_CM_scores_file_SSL_DF.txt'
truth_dir = '/kaggle/input/avsspoof-2021/DF-keys-full/keys/DF'
phase = 'eval'

cm_key_file = os.path.join(truth_dir, 'CM/trial_metadata.txt')

def eval_to_score_file(score_file, cm_key_file):

    # Load ground-truth metadata and submission scores
    cm_data = pandas.read_csv(cm_key_file, sep=' ', header=None)
    submission_scores = pandas.read_csv(score_file, sep=' ', header=None, skipinitialspace=True)

    # Check for column count consistency in submission file
    if len(submission_scores.columns) > 2:
        print(f'CHECK: submission has more columns ({len(submission_scores.columns)}) than expected (2).')
        print("Check for leading/ending blank spaces.")
        return None

    # Filter ground-truth data for the specific phase
    cm_data_phase = cm_data[cm_data[7] == phase]

    # Merge on common IDs, only keep available scores
    cm_scores = submission_scores.merge(cm_data_phase, left_on=0, right_on=1, how='inner')

    # Check if there are missing trials
    if len(cm_scores) < len(cm_data_phase):
        print(f"WARNING: Submission has {len(cm_scores)} of {len(cm_data_phase)} expected trials. Proceeding with available scores.")

    # Extract scores for bonafide and spoofed samples
    bona_cm = cm_scores[cm_scores[5] == 'bonafide']['1_x'].values
    spoof_cm = cm_scores[cm_scores[5] == 'spoof']['1_x'].values

    # Compute and print EER
    eer_cm = em.compute_eer(bona_cm, spoof_cm)[0]
    out_data = f"eer: {100 * eer_cm:.2f}\n"
    print(out_data)

    return eer_cm

if __name__ == "__main__":

    # Check that necessary files and directories exist
    if not os.path.isfile(submit_file):
        print(f"{submit_file} doesn't exist")
        sys.exit(1)

    if not os.path.isdir(truth_dir):
        print(f"{truth_dir} doesn't exist")
        sys.exit(1)

    # Verify that phase is valid
    if phase not in ['progress', 'eval', 'hidden_track']:
        print("phase must be either 'progress', 'eval', or 'hidden_track'")
        sys.exit(1)

    # Run evaluation
    _ = eval_to_score_file(submit_file, cm_key_file)



---

